# Modelforge ASE calculator 
## Simple Example

This notebook demonstrates the steps required to use a potential trained using modelforge as a calculator within the atomic simulation environment (ASE). 

First let us import some various modeules from modelforge we require for reading in a checkpoint file from a trained model

In [1]:
from modelforge.potential.potential import load_inference_model_from_checkpoint

# helper functions to load in the example model checkpoint file included in the repository
from modelforge.utils.io import get_path_string
from modelforge.ase.tests import data

# checkpoint file is saved in tests/data
checkpoint_file_path = get_path_string(data) + "/model.ckpt"
potential = load_inference_model_from_checkpoint(checkpoint_file_path, jit=False)


2026-04-19 22:27:20.533 | DEBUG    | modelforge.potential.potential:generate_potential:860 - training_parameter=None
2026-04-19 22:27:20.533 | DEBUG    | modelforge.potential.potential:generate_potential:861 - potential_parameter=SchNetParameters(potential_name='SchNet', only_unique_pairs=False, core_parameter=CoreParameter(number_of_radial_basis_functions=128, maximum_interaction_radius=0.49999999999999994, number_of_interaction_modules=9, number_of_filters=256, shared_interactions=True, activation_function_parameter=ActivationFunctionConfig(activation_function_name='ShiftedSoftplus', activation_function_arguments=None, activation_function=ShiftedSoftplus()), featurization=Featurization(properties_to_featurize=['atomic_number', 'per_system_total_charge'], atomic_number=AtomicNumber(maximum_atomic_number=101, number_of_per_atom_features=512), atomic_period=AtomicPeriod(maximum_period=8, number_of_per_period_features=32), atomic_group=AtomicGroup(maximum_group=18, number_of_per_group_fe

To use this potential in ASE, we will need to load the ModelForgeCalculator and pass it the potential on instantiation.

In [2]:
from modelforge.ase.calculator import ModelForgeCalculator

modelforge_calculator = ModelForgeCalculator(potential)


ASE has various built-in functions to set up molecules, but also a well defined API for manually setting these up.  As part of the modelforge.ase module, we have included some helper functions that will use accept a smiles string, use rdkit to initialize the 3d coordinates, and then initialize the configuration in a format ASE understands. 

In [3]:
from modelforge.ase.examples.helper_functions import smiles_to_ase, ase_to_rdkit

# note, optimize=True would use MMFF94 to optimize the structure within RDkit. 

smiles = "NCCCCCCO"
atoms = smiles_to_ase(smiles, optimize=False)
atoms.calc = modelforge_calculator



ASE provides a built in visualizer for viewing the 3d structure. Alternatively, we added a simple function to  `helper_functions.py` that will convert to rdkit. 

In [4]:
from ase.visualize import view
view(atoms, viewer='x3d')

### Single point energy and force computation
To extract the potential energy and per-atom forces on the system, we can use the following syntax. This will call the `ModelForgeCalculator` instance we set above.  Note, the calculator performs all appropriate unit conversions (modelforge internally uses kJ/mol and nanometers).  The units in ASE are eV and angstrom. 

In [5]:
# extract the energy adn forcesfrom ase.io import write, read
pe = atoms.get_potential_energy()
forces = atoms.get_forces()
print(f"potential energy: {pe}\n eV")
print(f"forces: {forces}\n eV/angstrom")

{'energy': -87.67781070908202, 'forces': array([[ 9.35616111e+00, -1.04942728e+01, -2.46964911e+00],
       [-4.73900153e+00,  2.68745397e+00, -1.49135413e+00],
       [ 1.76529851e+00, -4.81758771e-01,  1.01298696e+00],
       [-6.05731214e-01,  6.38253172e-01,  7.25336210e-02],
       [-8.12995186e-03, -3.55702845e-01, -7.89406278e-01],
       [ 1.21056213e+00,  8.26799015e-01, -6.81622256e-01],
       [ 6.57597244e-01,  4.57237875e-01, -7.60428314e-01],
       [-1.75304939e+00,  1.35703860e+00, -3.41916412e+00],
       [-8.86230649e+00, -1.88796916e+00,  4.35926526e+00],
       [ 3.55505226e+00,  7.28137009e+00,  4.92488868e-01],
       [ 6.69292667e-01,  6.24907588e-01, -1.99189454e+00],
       [ 2.48285644e-01,  2.60231257e+00,  7.58901380e-01],
       [-2.14170283e-01, -1.01459942e+00, -7.52638029e-01],
       [-9.50172969e-01, -6.46872713e-01,  9.01127572e-01],
       [-2.31541117e-01,  5.94767059e-01,  1.97118699e-01],
       [ 3.06847382e-01,  4.83846491e-01, -1.15195578e+00],

### System optimization 
ASE provides a host of functionality, such as optimization.

In [6]:
from ase.optimize import BFGS

opt = BFGS(atoms)
opt.run(fmax=0.05)

      Step     Time          Energy          fmax
BFGS:    0 22:27:20      -87.677811       14.274687
{'energy': -87.47746848295313, 'forces': array([[ 2.31028802, -5.67449669,  2.39000386],
       [ 2.41067219,  4.13026997, -0.9524969 ],
       [-5.8093775 , -0.5862369 , -0.68937705],
       [ 1.36116951,  1.38227558,  0.38892735],
       [ 0.43890142, -1.2772218 , -0.47571372],
       [-2.53929318, -1.08757494,  0.23727281],
       [ 0.6486697 ,  0.98310364,  1.85126202],
       [ 2.14360716, -2.41550666, -0.09089944],
       [ 1.41550907,  0.86866021, -0.11695552],
       [-0.59066506,  1.70773276,  0.81611864],
       [ 0.48686797,  0.80293126, -1.47067833],
       [ 0.39508364,  0.48840091,  0.16912199],
       [-0.91866523, -1.19677051, -0.90570686],
       [-0.40647444, -0.48621479,  0.82160891],
       [-0.04822451, -0.63521948, -0.38386481],
       [ 0.0491432 ,  0.0984644 , -0.19651142],
       [-0.99759484,  0.80992512, -0.57906522],
       [ 0.11076683,  0.05539027,  0.2472

np.True_

# System simulation
We can also perform simulations. 

In [7]:
import ase.units as ase_units
from ase.io.trajectory import Trajectory
from ase.md import Langevin
from ase.io import write, read

trajectory_name= f"{smiles}_sim.traj"
write(f'{smiles}_system.pdb', atoms)

traj = Trajectory(trajectory_name, "w", atoms)

dyn = Langevin(
    atoms,
    timestep=1.0* ase_units.fs,
    temperature_K=298,  # temperature in Kfrom ase.io import write, read
    friction=1.0/ ase_units.fs,
    trajectory=traj,
    logfile=f"{smiles}_md.log",
)
dyn.run(100)

# convert the ASE binary format to something vmd can read
traj = read(trajectory_name, ":")
write(f"{smiles}_sim.xyz", traj, format="xyz")

{'energy': -91.34972926395118, 'forces': array([[-0.86862099,  0.54844153,  0.13498146],
       [-0.3001805 , -1.85722034, -0.13673994],
       [-0.80682792, -0.76686759,  0.61409209],
       [ 1.27274284, -0.8703544 ,  0.93012844],
       [ 1.06749454,  0.28366826, -0.10066165],
       [ 0.14349143, -1.07441413,  0.95543361],
       [-2.04323413, -1.54532097, -1.1827504 ],
       [ 0.69629399, -0.59693891,  0.76539784],
       [ 0.39985596, -0.39912532, -0.12011849],
       [ 0.34815326, -0.90671406, -0.05295571],
       [ 0.57693531,  0.4586506 , -0.55957458],
       [-0.11012547,  2.11588792,  0.57875197],
       [-0.13557397,  1.72202767,  1.41804675],
       [ 1.07098514, -0.10968047, -2.51777669],
       [-0.39309811,  0.22207818, -0.01638919],
       [ 0.47540727,  0.19082256, -0.20086161],
       [-0.52529122,  0.67350062, -0.8007298 ],
       [-0.95958027, -0.53520266, -0.11902623],
       [-0.16688762, -0.50279605, -0.38255975],
       [-0.64766231,  0.84798226, -0.16566587],

/home/cri/miniconda3/envs/modelforge/lib/python3.13/site-packages/ase/md/langevin.py:110: FutureWarning: The implementation of `fixcm=True` in `Langevin` does not strictly sample the correct NVT distributions. The deviations are typically small for large systems but can be more pronounced for small systems. Use `fixcm=False` together with `ase.constraints.FixCom`. `fixcm` is deprecated since ASE 3.28.0 and will be removed in a future release.
  warnings.warn(msg, FutureWarning)


{'energy': -90.71392558804101, 'forces': array([[-1.99828352,  0.83089114, -3.64680543],
       [ 1.87288782, -3.18630627, -0.04619782],
       [ 2.1191909 , -2.15565918,  1.93647097],
       [ 0.76658356, -0.05984832,  4.91475796],
       [ 0.1656591 ,  0.02115792, -0.06630574],
       [-0.87706157, -1.09004519,  1.23320298],
       [-4.84580619,  1.40087447,  0.06212227],
       [ 3.04376511, -0.27993509,  4.03866507],
       [-1.75199588,  0.31995799,  0.66774543],
       [ 0.19181735,  0.88677377, -0.21143929],
       [-2.15080244, -0.84032115,  3.73420238],
       [ 0.77628134,  2.26572215,  0.06955415],
       [ 1.21957748, -1.64310736, -2.01116599],
       [ 1.73705055,  1.66361957,  0.16144462],
       [-0.28496962,  0.62074701,  0.10424438],
       [ 0.34730547,  1.7487306 , -4.13285168],
       [-0.69889847,  1.64116115, -1.09671567],
       [-0.44816872, -1.27850708, -0.18948757],
       [ 0.15156372, -0.11746884, -1.06879197],
       [ 1.02138035,  0.73696649, -0.49436768],

In [8]:
for atom in atoms:
    print(atom.number)
    print(atom.position)

7
[-3.24100426  0.04407142 -0.60753411]
6
[-2.73967138 -0.3400464  -0.0776647 ]
6
[-1.46272854  0.06178654  0.17402945]
6
[-0.29637411 -0.67522315  0.25990643]
6
[ 0.97571819 -0.16019589  0.3911978 ]
6
[ 1.53992357  0.18266099 -0.80650452]
6
[ 2.83554509  0.50449877 -0.404577  ]
8
[ 3.77738957 -0.02168205 -0.0100043 ]
1
[-4.58291604  0.07704931  0.07330512]
1
[-3.33099746  1.38105116 -0.16347694]
1
[-3.07060352 -0.48950482  0.73003526]
1
[-2.63189917 -1.55891243 -0.46440043]
1
[-1.65118936  0.99688357  0.29077915]
1
[-0.91608146  0.37736072 -1.07005986]
1
[-0.12776258 -1.91656669  0.0286428 ]
1
[-0.72677695 -1.20884214  1.47261904]
1
[ 1.63804523 -1.13902203  1.02291608]
1
[0.94294926 0.29275125 1.63809961]
1
[ 1.31897871 -0.79819697 -1.55843354]
1
[ 1.01964419  0.77931583 -1.11587065]
1
[3.18328708 1.66601241 0.2294726 ]
1
[ 3.50680707  1.05379047 -1.35899649]
1
[4.06459247 0.16658505 0.92917366]
